# CS234 RLAIF Evaluation Pipeline
**Direct vs Indirect GRPO — IFEval, AlpacaEval (Claude judge), Reward Hacking Diagnostic, Output Inspection**

Run cells top to bottom. Cell 1 (setup) and Cell 2 (generate outputs) require a GPU runtime.
All other cells can run on CPU.

**Required secrets** (set in Cell 1):
- `TOGETHER_API_KEY` — for base model generation
- `ANTHROPIC_API_KEY` — for AlpacaEval Claude judge

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from google.colab import drive, userdata

# ── API keys — fill these in ───────────────────────────────────────────────
os.environ["TOGETHER_API_KEY"]  = userdata.get("TOGETHER_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

# ── Drive paths — adjust if needed ────────────────────────────────────────
INDIRECT_ADAPTER = '/content/drive/MyDrive/cs234/grpo_indirect'
DIRECT_ADAPTER   = '/content/drive/MyDrive/cs234/grpo_direct'
DRIVE_RESULTS    = '/content/drive/MyDrive/cs234/eval_results'

OUTPUT_DIR  = './eval_outputs'
RESULTS_DIR = './eval_results'

drive.mount('/content/drive')

!git clone -b eval-gpt4o https://github.com/zcsabbagh/cs234-project.git /content/cs234-project 2>/dev/null \
  || git -C /content/cs234-project fetch origin eval-gpt4o && git -C /content/cs234-project checkout eval-gpt4o && git -C /content/cs234-project pull
%cd /content/cs234-project

!pip install -q torch transformers peft accelerate datasets \
             openai anthropic matplotlib numpy

import os
os.makedirs(OUTPUT_DIR,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Setup complete.")

In [ ]:
# ── Cell 2: Generate outputs (GPU required, ~1-2 hrs) ──────────────────────
# Generates responses from base (Together API), indirect LoRA, and direct LoRA
# on 805 AlpacaEval prompts + 541 IFEval prompts.
# Saves 6 JSONL files to eval_outputs/.
#
# Flags:
#   --skip-base       skip base model (if alpaca_base.jsonl already exists)
#   --skip-indirect   skip indirect model
#   --skip-direct     skip direct model

!python eval/generate_outputs.py \
    --indirect-adapter {INDIRECT_ADAPTER} \
    --direct-adapter   {DIRECT_ADAPTER} \
    --output-dir       {OUTPUT_DIR} \
    --n-alpaca         805 \
    --n-ifeval         541

In [ ]:
# ── Cell 3: IFEval (CPU ok, ~5 min) ───────────────────────────────────────
# Programmatic instruction-following eval — no judge needed.
# Outputs: ifeval_results.json, ifeval_table.txt, ifeval_bar.png

!python eval/run_ifeval.py \
    --output-dir  {OUTPUT_DIR} \
    --results-dir {RESULTS_DIR}

from IPython.display import Image
import os
_img = f'{RESULTS_DIR}/ifeval_bar.png'
Image(_img) if os.path.exists(_img) else print(f'Plot not found: {_img}')

In [ ]:
# ── Cell 4: AlpacaEval with Claude judge (CPU ok, ~30-40 min) ─────────────
# Compares each trained model vs GPT-4 Turbo reference outputs.
# Judge: Claude Sonnet (held-out — different from Llama 70B training judge).
# Outputs: alpacaeval_results.json, alpacaeval_table.txt, alpacaeval_bar.png

!python eval/run_alpacaeval_claude.py \
    --output-dir  {OUTPUT_DIR} \
    --results-dir {RESULTS_DIR} \
    --n-eval      805 \
    --workers     4

from IPython.display import Image
Image(f'{RESULTS_DIR}/alpacaeval_bar.png')

In [ ]:
# ── Cell 5: Training diagnostics + reward hacking plot (CPU ok, ~1 min) ───
# Reads trainer_state.json from each checkpoint.
# Outputs:
#   training_rewards.png   — reward curves over steps (both methods)
#   training_kl.png        — KL divergence over steps
#   reward_hacking.png     — training reward vs GPT-4o win rate (hacking diagnostic)
#   training_stats.txt     — printed + saved summary

!python eval/plot_diagnostics.py \
    --indirect-ckpt {INDIRECT_ADAPTER} \
    --direct-ckpt   {DIRECT_ADAPTER} \
    --results-dir   {RESULTS_DIR}

from IPython.display import Image, display
display(Image(f'{RESULTS_DIR}/training_rewards.png'))
display(Image(f'{RESULTS_DIR}/training_kl.png'))
display(Image(f'{RESULTS_DIR}/reward_hacking.png'))

In [ ]:
# ── Cell 6: Output inspection (CPU ok, ~2 min) ────────────────────────────
# Samples 75 outputs per model and flags degenerate patterns:
#   sycophantic openers, filler phrases, bullet overuse, length outliers.
# Outputs:
#   inspection_lengths.png    — length distribution
#   inspection_patterns.png   — pattern frequency bar chart
#   inspection_report.html    — human-readable report for manual review
#   inspection_summary.txt    — printed + saved table

!python eval/inspect_outputs.py \
    --output-dir  {OUTPUT_DIR} \
    --results-dir {RESULTS_DIR} \
    --n-samples   75

from IPython.display import Image, display
display(Image(f'{RESULTS_DIR}/inspection_lengths.png'))
display(Image(f'{RESULTS_DIR}/inspection_patterns.png'))

# Open HTML report in a scrollable iframe
from IPython.display import IFrame
IFrame(f'{RESULTS_DIR}/inspection_report.html', width='100%', height=600)

In [ ]:
# ── Cell 7: Print all results together ────────────────────────────────────
import json

print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)

# IFEval
ifeval_path = f'{RESULTS_DIR}/ifeval_results.json'
if os.path.exists(ifeval_path):
    with open(ifeval_path) as f:
        ifeval = json.load(f)
    print("\nIFEval (prompt-level strict accuracy):")
    for model, r in ifeval.items():
        print(f"  {model:<12} {r['prompt_strict']*100:.1f}% strict  {r['prompt_loose']*100:.1f}% loose")

# AlpacaEval
alpaca_path = f'{RESULTS_DIR}/alpacaeval_results.json'
if os.path.exists(alpaca_path):
    with open(alpaca_path) as f:
        alpaca = json.load(f)
    print("\nAlpacaEval win rate vs GPT-4 Turbo (Claude judge):")
    for model, r in alpaca.items():
        print(f"  {model:<12} {r['win_rate']*100:.1f}% ± {r['ci_95']*100:.1f}%")

# Inspection
insp_path = f'{RESULTS_DIR}/inspection_summary.json'
if os.path.exists(insp_path):
    with open(insp_path) as f:
        insp = json.load(f)
    print("\nOutput inspection (degenerate pattern rates):")
    for model, r in insp.items():
        pc = r['pattern_rates']
        print(f"  {model:<12} avg={r['avg_words']:.0f}w  "
              f"syco={pc['sycophantic']*100:.0f}%  "
              f"bullets={pc['bullet_heavy']*100:.0f}%  "
              f"filler={pc['filler_phrases']*100:.0f}%")

print("\n" + "="*60)

In [ ]:
# ── Cell 8: Save everything to Drive ──────────────────────────────────────
import shutil

if os.path.exists(DRIVE_RESULTS):
    shutil.rmtree(DRIVE_RESULTS)
shutil.copytree(RESULTS_DIR, DRIVE_RESULTS)

size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, _, files in os.walk(DRIVE_RESULTS) for f in files
) / 1e6
print(f"All results saved to Drive: {size:.1f} MB")
print(f"Location: {DRIVE_RESULTS}")
print("\nFiles saved:")
for f in sorted(os.listdir(DRIVE_RESULTS)):
    print(f"  {f}")